# Stretch: more questions, worked solutions

The answers to the stretch notebook. Try each one before looking.

In [ ]:
import os
import sqlite3
from pathlib import Path

here = Path.cwd()
while not (here / "data" / "music.db").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

con = sqlite3.connect("data/music.db")


def run(sql, limit=10):
    """Run a query and print the rows as an aligned table."""
    cursor = con.execute(sql)
    headers = [d[0] for d in cursor.description]
    rows = cursor.fetchall()
    widths = [max(len(str(h)), *(len(str(r[i])) for r in rows[:limit] or [[""]]))
              for i, h in enumerate(headers)]
    print("  ".join(str(h).ljust(w) for h, w in zip(headers, widths)))
    print("  ".join("-" * w for w in widths))
    for row in rows[:limit]:
        print("  ".join(str(v).ljust(w) for v, w in zip(row, widths)))
    print(f"({len(rows)} rows)" if len(rows) <= limit
          else f"... {len(rows)} rows in total")

## Stretch 1: the third table

The Festival Headline Award, 8 times. Six award names in total.

In [ ]:
run("""
SELECT award_name, COUNT(*) AS times_given
FROM awards
GROUP BY award_name
ORDER BY times_given DESC
""")

Six artists have three awards each, so there is no single winner. Two
counts side by side show who won the same award twice:

In [ ]:
run("""
SELECT artist_id,
       COUNT(*)                   AS awards,
       COUNT(DISTINCT award_name) AS different_awards
FROM awards
GROUP BY artist_id
HAVING COUNT(*) >= 3
ORDER BY artist_id
""")

Artists 1, 7 and 39 have three awards but only two different ones. If you
only want those three, the condition compares two aggregates, so it goes
in the `HAVING`:

In [ ]:
run("""
SELECT artist_id, COUNT(*) AS awards
FROM awards
GROUP BY artist_id
HAVING COUNT(*) > COUNT(DISTINCT award_name)
""")

In [ ]:
run("SELECT artist_id, artist_name FROM artists WHERE artist_id IN (1, 7, 39)")

Neon Harbour, The Quiet Wires and Tundra Post. It took two queries
because the names are in one table and the awards are in another.
Joining them in one query is what next session is for.

## Stretch 2: how many artists are still active?

Three ways to ask, and they give 39, 34 and 34:

In [ ]:
run("SELECT COUNT(still_active) AS active FROM artists")

In [ ]:
run("SELECT SUM(still_active) AS active FROM artists")

In [ ]:
run("SELECT COUNT(*) AS active FROM artists WHERE still_active = 1")

**The first one is wrong.** `COUNT(still_active)` does not count the
artists who are active. It counts the artists whose `still_active` is
**not NULL**, and that includes every `0`. "Is there a value?" and "is
the value 1?" are different questions.

The other two are both right. `SUM` works for the same reason as in the
homework skip rates: the column is already a 0 or a 1, so adding it up
counts the ones.

The second part:

In [ ]:
run("SELECT COUNT(*) AS not_active FROM artists WHERE still_active != 1")

34 active and 5 not active makes 39. There are 40 artists, so one is
missing from both answers.

In [ ]:
run("SELECT artist_name, still_active FROM artists WHERE still_active IS NULL")

Iron Fernway's `still_active` is unknown. Is unknown equal to 1? Unknown.
Is it different from 1? Also unknown. So `WHERE` drops the row both
times, and the artist disappears from "active" **and** from "not active".

That is how NULL catches people in real reports: two numbers that should
add up to the total, and do not. A count that does not add up is
always worth chasing.

## Stretch 3: artists by decade

In [ ]:
run("""
SELECT formed_year / 10 * 10 AS decade,
       COUNT(*)              AS artists
FROM artists
GROUP BY decade
ORDER BY decade
""")

The 2010s, with 22 of the 40 artists.

Walk through it with 1996: `1996 / 10` is `199`, because whole-number
division throws the fraction away, and `199 * 10` is `1990`. The thing
that ruins percentages is exactly what you want here.

**The `None` row.** Rosewood Lane and Cassette Revival have no
`formed_year`, and `NULL / 10` is still `NULL`. `GROUP BY` keeps them,
as one group of their own. That is honest: two artists we cannot place.
Filtering them out with `WHERE formed_year IS NOT NULL` is fine too, as
long as you say so, because then the counts add up to 38 and not 40.

## Stretch 4: tracks played on every device

35 of the 168 tracks have been played on all five devices.

In [ ]:
run("""
SELECT track_name,
       COUNT(DISTINCT device) AS devices,
       COUNT(*)               AS plays
FROM plays
GROUP BY track_name
HAVING COUNT(DISTINCT device) = 5
ORDER BY plays DESC
""")

Now only the plays that were not skipped. `skipped = 0` is a question
about **each play**, so it goes in the `WHERE`, before the grouping:

In [ ]:
run("""
SELECT track_name,
       COUNT(DISTINCT device) AS devices,
       COUNT(*)               AS plays
FROM plays
WHERE skipped = 0
GROUP BY track_name
HAVING COUNT(DISTINCT device) = 5
ORDER BY plays DESC
""")

30 tracks. Five drop out, because on at least one device they were only
ever skipped: Velvet Season, for example, was never listened to in the
car.

**The trap version.** Run this, with the condition moved to `HAVING`:

In [ ]:
run("""
SELECT track_name, COUNT(DISTINCT device) AS devices
FROM plays
GROUP BY track_name
HAVING COUNT(DISTINCT device) = 5 AND skipped = 0
""")

No error, 29 rows, and it looks like an answer. It is not. Blue Ferry
and Golden Garden are on the list, and both dropped out of the right
answer.

By `HAVING` time each track is **one group**, made of many plays. Some
were skipped and some were not, so "the group's `skipped`" means
nothing. SQLite picks one play's value and says nothing about it. That
is the `GROUP BY` trap from the slides, hiding in a `HAVING`.

The test: a condition about one row goes in `WHERE`. A condition about a
whole group (a count, an average, a sum) goes in `HAVING`.

## Stretch 5: do I listen more at the weekend?

In [ ]:
run("""
SELECT CASE
           WHEN strftime('%w', played_at) IN ('0', '6') THEN 'weekend'
           ELSE 'weekday'
       END                        AS part_of_week,
       COUNT(*)                   AS plays,
       COUNT(DISTINCT played_at)  AS days,
       ROUND(1.0 * COUNT(*) / COUNT(DISTINCT played_at), 2) AS plays_per_day
FROM plays
GROUP BY part_of_week
""")

Weekdays have 1,573 plays and weekends 610. That looks like a clear
answer: far more listening on weekdays.

**It is not a fair comparison.** A week has five weekdays and two weekend
days, so weekdays win on totals whatever the habits are. Per day it is
3.19 against 3.23: no difference at all.

Whenever two groups are different sizes, compare a rate, not a total.
Totals answer "where did most of it happen", rates answer "where is it
more likely". The question here was the second one.

Two details:

* `COUNT(DISTINCT played_at)` counts days **with at least one play**.
  Days with no listening do not appear in `plays`, so they cannot be
  counted. Fine here, worth knowing.
* `1.0 *` is the same fix as in the homework: without it, `1573 / 493`
  is `3`.

## Stretch 6: which device lost listeners?

In [ ]:
run("""
SELECT device,
       SUM(CASE WHEN played_at < '2025-01-01' THEN 1 ELSE 0 END)  AS plays_2024,
       SUM(CASE WHEN played_at >= '2025-01-01' THEN 1 ELSE 0 END) AS plays_2025,
       SUM(CASE WHEN played_at >= '2025-01-01' THEN 1 ELSE 0 END)
         - SUM(CASE WHEN played_at < '2025-01-01' THEN 1 ELSE 0 END) AS change
FROM plays
GROUP BY device
ORDER BY change
""")

The phone fell from 560 plays to 479, a drop of 81. Every other device
stayed level or grew a little.

That explains a number from the homework. There were 1,122 plays in 2024
and 1,061 in 2025, so 61 fewer overall. The phone alone lost 81, and the
other devices made up 20 of them. "Listening went down" is true, but
"phone listening went down" is the actual finding.

How the trick works: for each play, the `CASE` gives a 1 if the play is
in that year and a 0 if not. `SUM` adds those up per device, so it counts
only that year's plays. It is the same idea as `SUM(skipped)`, except
that you build the 0s and 1s yourself.

The change column has to repeat both `SUM(CASE ...)` expressions, because
a name made with `AS` cannot be used elsewhere in the same `SELECT`. It
can be used in `ORDER BY`, which runs later.

In [ ]:
con.close()
print("done")